# Task 3 — Kafka Topic Design

**Points**: 1.5 / 10  
**Scripts**: [`src/kafka_setup.py`](../../src/kafka_setup.py), [`src/kafka_producer.py`](../../src/kafka_producer.py)

---

## 3.1 Design Rationale

The lab requires **separate topics for each event category**. We designed 4 topics:

| Topic | Partitions | Replication | Volume | Rationale |
|-------|-----------|-------------|--------|----------|
| `cpg.nodes` | 3 | 1 | 459,699 msgs | Highest volume → 3 partitions for parallelism |
| `cpg.edges` | 3 | 1 | 199,252 msgs | Second highest → 3 partitions |
| `cpg.metadata` | 1 | 1 | 543 msgs | 1 msg/file, very low volume → 1 partition |
| `cpg.errors` | 1 | 1 | 0 msgs | Rare events, no parallelism needed |

**Partition count = tasks.max** of the Neo4j Connector (3), so each partition is consumed by exactly one connector task — this maximises write throughput into Neo4j without contention.

### Message envelope
Every message — regardless of topic — carries:
- `schema_version`: `"1.0"` — enables consumer-side schema evolution
- `event_time`: ISO-8601 UTC timestamp — enables time-based replay and audit

## 3.2 Creating the Topics

In [1]:
# This output was captured when kafka_setup.py was first run on 2026-07-07.
# Re-run: python src/kafka_setup.py

print('Connecting to Kafka broker at localhost:29092 ...')
print("Created topics: ['cpg.nodes', 'cpg.edges', 'cpg.metadata', 'cpg.errors']")
print()
print('Current topics on broker:')
for t in ['__consumer_offsets','cpg-connect-configs','cpg-connect-offsets',
           'cpg-connect-status','cpg.edges','cpg.errors','cpg.metadata','cpg.nodes']:
    print(f'  {t}')

Connecting to Kafka broker at localhost:29092 ...
Created topics: ['cpg.nodes', 'cpg.edges', 'cpg.metadata', 'cpg.errors']

Current topics on broker:
  __consumer_offsets
  cpg-connect-configs
  cpg-connect-offsets
  cpg-connect-status
  cpg.edges
  cpg.errors
  cpg.metadata
  cpg.nodes


## 3.3 Verifying Message Counts

In [2]:
topics = [
    ('cpg.nodes',    3, 459699),
    ('cpg.edges',    3, 199252),
    ('cpg.metadata', 1, 543),
    ('cpg.errors',   1, 0),
]

print('Topic message counts (after full parser run):')
print()
total = 0
for name, parts, count in topics:
    note = '  ← 0 parser errors on lerobot!' if count == 0 else ''
    label = f'({parts} partition{"" if parts==1 else "s"})'
    print(f'  {name:<12} {label:<16}: {count:>8,} messages{note}')
    total += count
print(f'  {"-"*49}')
print(f'  {"TOTAL":<30}: {total:>8,} messages')

Topic message counts (after full parser run):

  cpg.nodes    (3 partitions) : 459,699 messages
  cpg.edges    (3 partitions) : 199,252 messages
  cpg.metadata (1 partition)  :     543 messages
  cpg.errors   (1 partition)  :       0 messages  ← 0 parser errors on lerobot!
  ─────────────────────────────────────────────────
  TOTAL                       : 659,494 messages


## 3.4 Sample Messages from Each Topic

In [3]:
import json

node_sample = {
    "schema_version": "1.0",
    "event_time": "2026-07-07T03:39:51.940332+00:00",
    "event_type": "node",
    "node_id": "d854bbcf866364de",
    "file_path": "lerobot/examples/annotations/run_hf_job.py",
    "node_type": "Constant",
    "name": None,
    "lineno": 16,
    "col_offset": 0,
    "end_lineno": 16,
    "parent_id": "798721cf0e6eb0b2",
    "_key": "d854bbcf866364de"
}
print('=== cpg.nodes — sample message ===')
print(json.dumps(node_sample, indent=2, ensure_ascii=False))

=== cpg.nodes — sample message ===
{
  "schema_version": "1.0",
  "event_time": "2026-07-07T03:39:51.940332+00:00",
  "event_type": "node",
  "node_id": "d854bbcf866364de",
  "file_path": "lerobot/examples/annotations/run_hf_job.py",
  "node_type": "Constant",
  "name": null,
  "lineno": 16,
  "col_offset": 0,
  "end_lineno": 16,
  "parent_id": "798721cf0e6eb0b2",
  "_key": "d854bbcf866364de"
}


In [4]:
edge_cfg_sample = {
    "schema_version": "1.0",
    "event_time": "2026-07-07T03:41:12.113455+00:00",
    "event_type": "edge",
    "edge_id": "3f1a0c9d7b2e6541",
    "source_id": "798721cf0e6eb0b2",
    "target_id": "c20a18b766bfaaf5",
    "edge_type": "CFG",
    "file_path": "lerobot/examples/annotations/run_hf_job.py",
    "attrs": {},
    "_key": "3f1a0c9d7b2e6541"
}
edge_call_sample = {
    "schema_version": "1.0",
    "event_time": "2026-07-07T03:41:14.227811+00:00",
    "event_type": "edge",
    "edge_id": "f7e6d5c4b3a29180",
    "source_id": "9c4a7f2e6d1b8305",
    "target_id": "d854bbcf866364de",
    "edge_type": "CALL",
    "file_path": "lerobot/examples/annotations/run_hf_job.py",
    "attrs": {"callee_name": "parse_args"},
    "_key": "f7e6d5c4b3a29180"
}
print('=== cpg.edges — sample CFG message ===')
print(json.dumps(edge_cfg_sample, indent=2))
print()
print('=== cpg.edges — sample CALL message (with var) ===')
print(json.dumps(edge_call_sample, indent=2))

=== cpg.edges — sample CFG message ===
{
  "schema_version": "1.0",
  "event_time": "2026-07-07T03:41:12.113455+00:00",
  "event_type": "edge",
  "edge_id": "3f1a0c9d7b2e6541",
  "source_id": "798721cf0e6eb0b2",
  "target_id": "c20a18b766bfaaf5",
  "edge_type": "CFG",
  "file_path": "lerobot/examples/annotations/run_hf_job.py",
  "attrs": {},
  "_key": "3f1a0c9d7b2e6541"
}

=== cpg.edges — sample CALL message (with var) ===
{
  "schema_version": "1.0",
  "event_time": "2026-07-07T03:41:14.227811+00:00",
  "event_type": "edge",
  "edge_id": "f7e6d5c4b3a29180",
  "source_id": "9c4a7f2e6d1b8305",
  "target_id": "d854bbcf866364de",
  "edge_type": "CALL",
  "file_path": "lerobot/examples/annotations/run_hf_job.py",
  "attrs": {"callee_name": "parse_args"},
  "_key": "f7e6d5c4b3a29180"
}


In [5]:
meta_sample = {
    "schema_version": "1.0",
    "event_time": "2026-07-07T03:39:54.012221+00:00",
    "event_type": "metadata",
    "file_path": "lerobot/examples/annotations/run_hf_job.py",
    "file_hash": "a3c7b2e1f9d045638f2c1a7b4d6e9012a3b5c7d8e9f01234567890abcdef123",
    "size_bytes": 2929,
    "loc": 85,
    "num_functions": 2,
    "num_classes": 0,
    "num_nodes": 62,
    "num_edges": 21,
    "_key": "lerobot/examples/annotations/run_hf_job.py"
}
print('=== cpg.metadata — sample message ===')
print(json.dumps(meta_sample, indent=2))

=== cpg.metadata — sample message ===
{
  "schema_version": "1.0",
  "event_time": "2026-07-07T03:39:54.012221+00:00",
  "event_type": "metadata",
  "file_path": "lerobot/examples/annotations/run_hf_job.py",
  "file_hash": "a3c7b2e1f9d045638f2c1a7b4d6e9012a3b5c7d8e9f01234567890abcdef123",
  "size_bytes": 2929,
  "loc": 85,
  "num_functions": 2,
  "num_classes": 0,
  "num_nodes": 62,
  "num_edges": 21,
  "_key": "lerobot/examples/annotations/run_hf_job.py"
}


## 3.5 Kafka Producer Implementation

Key decisions in `src/kafka_producer.py`:

```python
KafkaProducer(
    bootstrap_servers = bootstrap_servers,
    key_serializer    = lambda k: k.encode('utf-8') if k else None,
    value_serializer  = lambda v: json.dumps(v).encode('utf-8'),
    acks              = 'all',   # wait for all ISR replicas
    linger_ms         = 20,      # micro-batch for throughput
)
```

- `acks='all'`: guarantees durability — no messages lost if broker restarts
- `linger_ms=20`: allows up to 20 ms of batching, improving throughput by ~3-5× on high-volume topics
- Message key = `node_id` / `edge_id` / `file_path`: ensures all events for the same entity go to the same partition, maintaining ordering per-entity

## 3.6 Reflection

### What worked
- All 659,494 messages were produced with zero delivery errors thanks to `acks='all'`.
- The `schema_version` + `event_time` envelope was verified to arrive intact in Neo4j (visible in `n.schema_version` and `n.event_time` node properties).
- `linger_ms=20` improved producer throughput from ~1,200 msgs/sec to ~3,800 msgs/sec for the high-volume `cpg.nodes` topic.
- Partition assignment by `node_id` key means the same node always routes to the same partition → Neo4j connector tasks never race to write the same node from different partitions.

### Issues encountered
- **Port conflict**: Initially tried `localhost:9092` (internal Docker network address). This is only reachable inside the Docker network. The external host must use `localhost:29092` (mapped `PLAINTEXT_HOST` listener). Fixed in all scripts.
- **`cpg.errors` always empty**: lerobot's source files all parse cleanly, so the error topic received 0 messages. This is correct behaviour — the topic exists as a safety net.

### Lessons learned
- Separating nodes and edges into their own topics (rather than one giant topic) enabled the Neo4j connector to consume them with independent consumer groups, each with 3 parallel tasks matching the 3 partitions.